In [12]:
!pip install --upgrade pip setuptools wheel
!pip install "numpy<2" "pandas<2.2" "scipy<1.11" "scikit-learn<1.4" "lightgbm==3.4.1"
!pip install pycaret

import pandas as pd
from pycaret.classification import *
from sklearn.model_selection import train_test_split

SEED = 128

  Using cached scipy-1.10.1-cp39-cp39-macosx_12_0_arm64.whl.metadata (53 kB)
  Using cached scikit_learn-1.3.2-cp39-cp39-macosx_12_0_arm64.whl.metadata (11 kB)
ERROR: Ignored the following versions that require a different python version: 1.14.0 Requires-Python >=3.10; 1.14.0rc1 Requires-Python >=3.10; 1.14.0rc2 Requires-Python >=3.10; 1.14.1 Requires-Python >=3.10; 1.15.0 Requires-Python >=3.10; 1.15.0rc1 Requires-Python >=3.10; 1.15.0rc2 Requires-Python >=3.10; 1.15.1 Requires-Python >=3.10; 1.15.2 Requires-Python >=3.10; 1.15.3 Requires-Python >=3.10; 1.16.0 Requires-Python >=3.11; 1.16.0rc1 Requires-Python >=3.11; 1.16.0rc2 Requires-Python >=3.11; 1.16.1 Requires-Python >=3.11; 1.16.2 Requires-Python >=3.11; 1.16.3 Requires-Python >=3.11; 1.7.0 Requires-Python >=3.10; 1.7.0rc1 Requires-Python >=3.10; 1.7.1 Requires-Python >=3.10; 1.7.2 Requires-Python >=3.10
ERROR: Could not find a version that satisfies the requirement lightgbm==3.4.1 (from versions: 2.0.2, 2.0.3, 2.0.4, 2.0.5, 2.

In [ ]:
df = pd.read_csv("../Dataset1_UK_Housing/5_price_paid_records_final.csv")
df_sample = df.sample(50000, random_state=SEED) # Take a sample from dataset to train faster


In [14]:
train_df, val_df = train_test_split(
    df_sample, 
    test_size=0.3, 
    random_state=SEED, 
    stratify=df_sample["property_type"]
)

clf = setup(
    data=train_df,
    target="property_type",
    session_id=SEED,
    fold=2,
    verbose=True
)

best_model = compare_models(sort="F1", n_select=1, turbo=True)

tuned_model = tune_model(
    best_model, 
    optimize="F1", 
    fold=5, 
    n_iter=20
)

final_model = finalize_model(tuned_model)

predictions = predict_model(final_model, data=val_df)

from sklearn.metrics import accuracy_score, f1_score

def eval_preds(pred):
    y_true = pred["property_type"]
    y_pred = pred["prediction_label"]
    return accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")

eval_final = eval_preds(predictions)
print(eval_final)

save_model(final_model, "../Model_UKHousing/best_property_type_model")

,Description,Value
0,Session id,128
1,Target,property_type
2,Target type,Multiclass
3,Target mapping,"D: 0, F: 1, O: 2, S: 3, T: 4"
4,Original data shape,"(35000, 10)"
5,Transformed data shape,"(35000, 10)"
6,Transformed train set shape,"(24500, 10)"
7,Transformed test set shape,"(10500, 10)"
8,Numeric features,2
9,Categorical features,7


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.5561,0.7773,0.5561,0.5548,0.5553,0.4051,0.4051,3.1250
rf,Random Forest Classifier,0.5399,0.7748,0.5399,0.5370,0.5378,0.3832,0.3833,0.6300
gbc,Gradient Boosting Classifier,0.5400,0.0000,0.5400,0.5334,0.5362,0.3851,0.3852,2.4250
et,Extra Trees Classifier,0.5366,0.7673,0.5366,0.5305,0.5332,0.3800,0.3801,0.2150
lr,Logistic Regression,0.5142,0.0000,0.5142,0.4973,0.4993,0.3485,0.3509,1.2600
dt,Decision Tree Classifier,0.4832,0.6475,0.4832,0.4890,0.4858,0.3057,0.3059,0.5450
lda,Linear Discriminant Analysis,0.4709,0.0000,0.4709,0.4527,0.4560,0.2983,0.3000,0.5450
ridge,Ridge Classifier,0.4719,0.0000,0.4719,0.4554,0.4160,0.2936,0.3107,0.5450
knn,K Neighbors Classifier,0.3895,0.6341,0.3895,0.3951,0.3870,0.1826,0.1841,0.6050
ada,Ada Boost Classifier,0.4619,0.0000,0.4619,0.4204,0.3758,0.3069,0.3715,0.6450


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.5904,0.8124,0.5904,0.5841,0.5868,0.4525,0.4527
1,0.6012,0.8148,0.6012,0.5959,0.5983,0.4660,0.4661
2,0.5957,0.8169,0.5957,0.5880,0.5912,0.4580,0.4583
3,0.5984,0.8142,0.5984,0.5940,0.5960,0.4619,0.4619
4,0.5943,0.8110,0.5943,0.5863,0.5897,0.4573,0.4575
Mean,0.5960,0.8139,0.5960,0.5897,0.5924,0.4591,0.4593
Std,0.0037,0.0020,0.0037,0.0046,0.0042,0.0045,0.0045


Fitting 5 folds for each of 20 candidates, totalling 100 fits


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Light Gradient Boosting Machine,0.6369,0.8468,0.6369,0.6294,0.6326,0.5139,0.5142


[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
[LightGBM] [Warning] bagging_freq is set=3, subsample_freq=0 will be ignored. Current value: bagging_freq=3
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] bagging_fraction is set=1.0, subsample=1.0 will be ignored. Current value: bagging_fraction=1.0
(0.6368666666666667, 0.6272424954399549)
Transformation Pipeline and Model Successfully Saved


(Pipeline(memory=Memory(location=None),
          steps=[('label_encoding',
                  TransformerWrapperWithInverse(exclude=None, include=None,
                                                transformer=LabelEncoder())),
                 ('numerical_imputer',
                  TransformerWrapper(exclude=None, include=['price', 'year'],
                                     transformer=SimpleImputer(add_indicator=False,
                                                               copy=True,
                                                               fill_value=None,
                                                               keep_empty_features=False,
                                                               missing_values=nan,
                                                               strateg...
                                 boosting_type='gbdt', class_weight=None,
                                 colsample_bytree=1.0, feature_fraction=0.6,
                 